# Run FedRBE on Quartet proteomics multi-batch dataset

This notebook runs the federated batch effect correction (FedRBE) on the proteomics multi-batch dataset
(Quartet, PXD045065) using the FeatureCloud testing environment.

**Prerequisites:**
- Docker running
- `featurecloud.ai/bcorrect:latest` image available (pull with `docker pull featurecloud.ai/bcorrect:latest`)
- Data prepared by `01_data_prep_RBE.ipynb` (center folders in `before/`)
- Python packages: `FeatureCloud`, `pandas`, `pyyaml` (from `requirements.txt`)

**What this does:**
1. Defines the experiment for 4 centers (APT, FDU, NVG, BGI) with per-center config
2. Runs FedRBE via the FeatureCloud controller (non-SMPC and SMPC variants)
3. Extracts and concatenates per-client results into `after/FedApp_corrected_data.tsv` and `after/FedApp_corrected_data_smpc.tsv`

This mirrors the logic in `generate_fedrbe_corrected_datasets.py` but is self-contained for the multi-batch dataset.

In [83]:
import os
import json
import zipfile
import time
import pandas as pd
from copy import deepcopy

import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
from evaluation_utils import featurecloud_api_extension as util

## Settings

In [84]:
# Paths
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
data_dir = os.path.join(repo_root, 'evaluation_data')
multibatch_dir = os.path.join(data_dir, 'proteomics_multibatch')
before_dir = os.path.join(multibatch_dir, 'before')
after_dir = os.path.join(multibatch_dir, 'after')
os.makedirs(after_dir, exist_ok=True)

app_image_name = 'featurecloud.ai/bcorrect:latest'

# Centers and their properties
# APT and FDU have 2 internal batches (DDA + DIA)
# NVG and BGI have 1 batch each (single batch name for all samples)
# All centers use batch_col: batch
centers = ['APT', 'FDU', 'NVG', 'BGI']

print(f'Data dir: {data_dir}')
print(f'Before dir: {before_dir}')
print(f'After dir: {after_dir}')
print(f'Centers: {centers}')

Data dir: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data
Before dir: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/before
After dir: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/after
Centers: ['APT', 'FDU', 'NVG', 'BGI']


## Define experiment configuration

The base config matches the FedRBE app's expected `config.yml` format.
Per-center overrides set the correct `batch_col` (all centers use \`batch\`).

In [85]:
base_config = {
    'flimmaBatchCorrection': {
        'data_filename': 'data.tsv',
        'design_filename': 'design.tsv',
        'expression_file_flag': True,
        'index_col': 0,
        'batch_col': None,
        'covariates': [],
        'separator': '\t',
        'design_separator': '\t',
        'normalizationMethod': None,
        'smpc': False,
        'min_samples': 0,
        'position': None,
        'reference_batch': False,
    },
}

# Common overrides for all centers
common_changes = {
    'flimmaBatchCorrection': {
        'data_filename': 'intensities_log_UNION.tsv',
        'design_filename': 'design.tsv',
        'covariates': ['D6', 'F7', 'M8'],
        'index_col': 'rowname',
        'min_samples': 0,
    }
}

# Build per-center config changes.
# BGI is the reference batch (last in the centers list, highest position).
# All centers use batch_col: batch (single-batch centers have one batch name for all samples).
config_file_changes = []
for center in centers:
    changes = deepcopy(common_changes)
    changes['flimmaBatchCorrection']['batch_col'] = 'batch'
    # Explicitly mark BGI as the reference batch, others as non-reference
    changes['flimmaBatchCorrection']['reference_batch'] = (center == 'BGI')
    config_file_changes.append(changes)

print('Per-center batch_col:')
for center, changes in zip(centers, config_file_changes):
    bc = changes['flimmaBatchCorrection']
    print(f"  {center}: batch_col={bc['batch_col']}, reference_batch={bc['reference_batch']}")

Per-center batch_col:
  APT: batch_col=batch, reference_batch=False
  FDU: batch_col=batch, reference_batch=False
  NVG: batch_col=batch, reference_batch=False
  BGI: batch_col=batch, reference_batch=True


## Helper: postprocessing

Extracts per-client results from ZIP files, concatenates into a single corrected matrix.

In [ ]:
# Use the shared helper from evaluation_utils to keep this in sync with multiomics.
from evaluation_utils.featurecloud_api_extension import postprocess_results

## Run FedRBE (non-SMPC)

In [87]:
experiment = util.Experiment(
    name='Proteomics Multi Batch',
    fc_data_dir=multibatch_dir,
    clients=[os.path.join(before_dir, c) for c in centers],
    app_image_name=app_image_name,
    config_files=[deepcopy(base_config) for _ in centers],
    config_file_changes=deepcopy(config_file_changes),
)

# Add position argument (required for federated coordination)
for idx in range(len(experiment.clients)):
    experiment.config_file_changes[idx]['flimmaBatchCorrection']['position'] = idx

print(experiment)

Experiment(name='Proteomics Multi Batch', clients=['/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/before/APT', '/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/before/FDU', '/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/before/NVG', '/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/before/BGI'], app_image_name='featurecloud.ai/bcorrect:latest', fc_data_dir='/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch', config_files=[{'flimmaBatchCorrection': {'data_filename': 'data.tsv', 'design_filename': 'design.tsv', 'expression_file_flag': True, 'index_col': 0, 'batch_col': None, 'covariates': [], 'separator': '\t', 'design_separator': '\t', 'normalizationMethod': None, 'smpc': False, 'min_samples': 0, 'position': None, 'reference_batch': False}}, {'flimmaBatchCorrection': {'data_filename': 'data.tsv', 'design_filename': 'design.tsv',

In [88]:
# print experiment config for debugging
for idx, (client, config) in enumerate(zip(experiment.clients, experiment.config_file_changes)):
    print(f"\nClient {idx} ({os.path.basename(client)}):")
    print(json.dumps(config, indent=2))


Client 0 (APT):
{
  "flimmaBatchCorrection": {
    "data_filename": "intensities_log_UNION.tsv",
    "design_filename": "design.tsv",
    "covariates": [
      "D6",
      "F7",
      "M8"
    ],
    "index_col": "rowname",
    "min_samples": 0,
    "batch_col": "batch",
    "reference_batch": false,
    "position": 0
  }
}

Client 1 (FDU):
{
  "flimmaBatchCorrection": {
    "data_filename": "intensities_log_UNION.tsv",
    "design_filename": "design.tsv",
    "covariates": [
      "D6",
      "F7",
      "M8"
    ],
    "index_col": "rowname",
    "min_samples": 0,
    "batch_col": "batch",
    "reference_batch": false,
    "position": 1
  }
}

Client 2 (NVG):
{
  "flimmaBatchCorrection": {
    "data_filename": "intensities_log_UNION.tsv",
    "design_filename": "design.tsv",
    "covariates": [
      "D6",
      "F7",
      "M8"
    ],
    "index_col": "rowname",
    "min_samples": 0,
    "batch_col": "batch",
    "reference_batch": false,
    "position": 2
  }
}

Client 3 (BGI):
{


In [89]:
result_files_zipped, _, _ = experiment.run_test()
print(f'Result files: {result_files_zipped}')

_______________EXPERIMENT_______________
Killing leftover containers...
Controller not running, starting it now!
Killing leftover containers...
Starting the FeatureCloud controller with the data directory /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch


Downloading...: 3it [00:00, 876.43it/s]


Controller started successfully, waiting 5s to ensure the controller is properly running!
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_featurecloudaibcorrectlatest_797783865', 'containerID': '8c33a6d92d50e871', 'coordinator': True, 'frontendUrl': '/app-traffic/6e5093aed8/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_featurecloudaibcorrectlatest_20305154', 'containerID': '588f7ecd0e46f826', 'coordinator': False, 'frontendUrl': '/app-traffic/09cc87b420/', 'message': 'terminal', 'state': None, 'progress': None}, {'id': 2, 'name': 'fc_featurecloudaibcorrectlatest_234172446', 'containerID': '1ba7cce74318d365', 'coordinator': False, 'frontendUrl': '/app-traffic/70c0387f33/', 'message': 'terminal', 'state': None, 'progress': None}, {'id': 3, 'name': 'fc_featurecloudaibcorrectlatest_425134484', 'containerID': '2bde0d75ed563796', 'coordinator': False, 'frontendUrl': '/app-traffic/8e0f240556/', 'message': 'terminal', 's

In [90]:
result_path = os.path.join(after_dir, 'FedApp_corrected_data.tsv')
corrected_df = postprocess_results(experiment, result_files_zipped, result_path)
corrected_df.head()

Saved concatenated result ((3407, 72)) to /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/after/FedApp_corrected_data.tsv


,DDA_APT_D5_1,DDA_APT_D6_1,DDA_APT_F7_1,DDA_APT_M8_1,DDA_APT_D5_2,DDA_APT_D6_2,DDA_APT_F7_2,DDA_APT_M8_2,DDA_APT_D5_3,DDA_APT_D6_3,DDA_APT_F7_3,DDA_APT_M8_3,DIA_APT_D5_1,DIA_APT_D6_1,DIA_APT_F7_1,DIA_APT_M8_1,DIA_APT_D5_2,DIA_APT_D6_2,DIA_APT_F7_2,DIA_APT_M8_2,DIA_APT_D5_3,DIA_APT_D6_3,DIA_APT_F7_3,DIA_APT_M8_3,DDA_FDU_D5_1,DDA_FDU_D6_1,DDA_FDU_F7_1,DDA_FDU_M8_1,DDA_FDU_D5_2,DDA_FDU_D6_2,DDA_FDU_F7_2,DDA_FDU_M8_2,DDA_FDU_D5_3,DDA_FDU_D6_3,DDA_FDU_F7_3,DDA_FDU_M8_3,DIA_FDU_D5_1,DIA_FDU_D6_1,DIA_FDU_F7_1,DIA_FDU_M8_1,DIA_FDU_D5_2,DIA_FDU_D6_2,DIA_FDU_F7_2,DIA_FDU_M8_2,DIA_FDU_D5_3,DIA_FDU_D6_3,DIA_FDU_F7_3,DIA_FDU_M8_3,DDA_NVG_D5_1,DDA_NVG_D6_1,DDA_NVG_F7_1,DDA_NVG_M8_1,DDA_NVG_D5_2,DDA_NVG_D6_2,DDA_NVG_F7_2,DDA_NVG_M8_2,DDA_NVG_D5_3,DDA_NVG_D6_3,DDA_NVG_F7_3,DDA_NVG_M8_3,DIA_BGI_D5_1,DIA_BGI_D6_1,DIA_BGI_F7_1,DIA_BGI_M8_1,DIA_BGI_D5_2,DIA_BGI_D6_2,DIA_BGI_F7_2,DIA_BGI_M8_2,DIA_BGI_D5_3,DIA_BGI_D6_3,DIA_BGI_F7_3,DIA_BGI_M8_3
rowname,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Asparagine synthetase [glutamine-hydrolyzing],19.860845,19.991346,20.145889,19.806990,19.967013,19.970993,20.093606,19.614526,19.995608,20.100427,20.086355,19.869796,19.775381,19.839469,20.209254,19.577877,19.803350,20.077635,20.342796,19.759229,19.909174,20.171098,20.099340,19.938791,19.891372,20.064317,20.181326,19.720069,19.803541,20.014973,20.275512,19.807146,19.976571,20.035710,20.141858,19.590997,19.684810,20.054783,20.251451,19.744597,19.909789,20.068129,20.310009,19.665728,19.856194,20.014483,20.235010,19.708410,19.821773,19.921400,20.153889,19.807108,19.896960,19.645544,20.329848,19.809738,20.174417,20.004120,20.221917,19.716678,19.913287,20.129740,20.225547,19.694733,19.920845,19.894252,20.270235,19.624782,19.903246,20.056401,20.243377,19.626948
7-dehydrocholesterol reductase,18.558493,18.917571,18.991289,18.684918,18.662426,19.032271,18.860557,18.776803,18.786864,18.800642,18.801431,19.100517,18.839435,18.531113,19.091305,18.845724,18.814530,18.784240,19.031933,18.789758,18.753161,18.570840,19.038279,18.883462,18.719959,18.756354,18.988160,18.724251,18.780447,18.896590,18.842797,18.833307,18.756866,18.751393,18.967301,18.956356,18.675268,18.808181,18.967849,18.892759,18.830904,18.606145,18.903201,18.899269,18.775478,18.866127,18.898324,18.850276,18.683514,18.480757,18.859581,18.580172,18.529426,19.271028,19.027993,18.768975,18.807566,19.001903,18.993718,18.969148,18.594127,18.767989,18.906309,18.685014,18.699358,18.930630,19.169211,18.738473,18.543816,18.986024,18.912861,19.039970
Glycolipid transfer protein,17.983987,18.928274,18.020263,19.044653,17.852047,18.577714,17.479855,18.915431,17.890074,18.969857,18.940817,18.783962,17.492014,18.751758,18.330559,18.941449,18.162516,18.723220,18.382085,18.758840,18.584169,18.153339,18.701750,18.405237,18.398640,18.588739,18.270514,18.592130,18.193263,18.321516,18.477697,18.834651,18.359653,18.292609,18.513505,18.544018,18.271716,18.361208,18.414510,18.827920,18.396416,18.503304,18.539536,18.829533,18.214655,18.360058,18.139170,18.528908,18.333994,18.297549,17.734340,19.239760,NaN,18.196135,18.497873,18.457580,18.243017,19.292838,18.182250,18.727757,18.002655,18.621614,18.356020,18.532285,18.375956,18.465961,18.227445,18.867166,18.370531,18.224416,18.542546,18.800341
"39S ribosomal protein L44, mitochondrial",19.098223,18.658105,18.812193,18.660543,18.778211,18.610128,18.676340,18.583907,19.338983,18.658374,18.883156,18.526606,19.179880,18.942425,18.468377,18.488505,19.441536,18.823241,18.230636,18.761014,18.920199,18.750227,18.749834,18.528896,18.800213,18.675607,18.640373,18.963697,18.776979,18.722274,18.608510,18.699248,19.253732,18.985702,18.503170,18.655266,18.701337,19.181015,18.649929,18.671866,19.089580,18.739683,18.771583,18.447250,19.105507,18.721854,18.793255,18.411912,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.850272,18.797546,18.703006,18.687197,19.057396,18.734226,18.796260,18.571871,19.017570,18.786674,18.623568,18.659185
Protein deglycase DJ-1,21.501443,

## Run FedRBE (SMPC)

Same experiment but with Secure Multi-Party Computation enabled.

In [91]:
experiment_smpc = util.Experiment(
    name='Proteomics Multi Batch (SMPC)',
    fc_data_dir=multibatch_dir,
    clients=[os.path.join(before_dir, c) for c in centers],
    app_image_name=app_image_name,
    config_files=[deepcopy(base_config) for _ in centers],
    config_file_changes=deepcopy(config_file_changes),
)

# Enable SMPC and set position
for idx in range(len(experiment_smpc.clients)):
    experiment_smpc.config_file_changes[idx]['flimmaBatchCorrection']['smpc'] = True
    experiment_smpc.config_file_changes[idx]['flimmaBatchCorrection']['position'] = idx

print(experiment_smpc)

Experiment(name='Proteomics Multi Batch (SMPC)', clients=['/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/before/APT', '/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/before/FDU', '/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/before/NVG', '/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/before/BGI'], app_image_name='featurecloud.ai/bcorrect:latest', fc_data_dir='/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch', config_files=[{'flimmaBatchCorrection': {'data_filename': 'data.tsv', 'design_filename': 'design.tsv', 'expression_file_flag': True, 'index_col': 0, 'batch_col': None, 'covariates': [], 'separator': '\t', 'design_separator': '\t', 'normalizationMethod': None, 'smpc': False, 'min_samples': 0, 'position': None, 'reference_batch': False}}, {'flimmaBatchCorrection': {'data_filename': 'data.tsv', 'design_filename': 'desig

In [92]:
result_files_zipped_smpc, _, _ = experiment_smpc.run_test()
print(f'Result files: {result_files_zipped_smpc}')

_______________EXPERIMENT_______________
Killing leftover containers...
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_featurecloudaibcorrectlatest_834638687', 'containerID': '75749e89beb9066f', 'coordinator': False, 'frontendUrl': '/app-traffic/1c7e7a53a1/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_featurecloudaibcorrectlatest_310190042', 'containerID': '22a6de09ef59b470', 'coordinator': False, 'frontendUrl': '/app-traffic/48b0ffbfac/', 'message': 'terminal', 'state': None, 'progress': None}, {'id': 2, 'name': 'fc_featurecloudaibcorrectlatest_704890821', 'containerID': '0dd4beb54a53f495', 'coordinator': False, 'frontendUrl': '/app-traffic/bd27d4a8ec/', 'message': 'terminal', 'state': None, 'progress': None}, {'id': 3, 'name': 'fc_featurecloudaibcorrectlatest_119455194', 'containerID': '408760df22f53d18', 'coordinator': True, 'frontendUrl': '/app-traffic/3bc7829da7/', 'message': 'terminal', 'state': None, 'pro

In [93]:
result_path_smpc = os.path.join(after_dir, 'FedApp_corrected_data_smpc.tsv')
corrected_df_smpc = postprocess_results(experiment_smpc, result_files_zipped_smpc, result_path_smpc)
corrected_df_smpc.head()

Saved concatenated result ((3407, 72)) to /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/after/FedApp_corrected_data_smpc.tsv


,DDA_APT_D5_1,DDA_APT_D6_1,DDA_APT_F7_1,DDA_APT_M8_1,DDA_APT_D5_2,DDA_APT_D6_2,DDA_APT_F7_2,DDA_APT_M8_2,DDA_APT_D5_3,DDA_APT_D6_3,DDA_APT_F7_3,DDA_APT_M8_3,DIA_APT_D5_1,DIA_APT_D6_1,DIA_APT_F7_1,DIA_APT_M8_1,DIA_APT_D5_2,DIA_APT_D6_2,DIA_APT_F7_2,DIA_APT_M8_2,DIA_APT_D5_3,DIA_APT_D6_3,DIA_APT_F7_3,DIA_APT_M8_3,DDA_FDU_D5_1,DDA_FDU_D6_1,DDA_FDU_F7_1,DDA_FDU_M8_1,DDA_FDU_D5_2,DDA_FDU_D6_2,DDA_FDU_F7_2,DDA_FDU_M8_2,DDA_FDU_D5_3,DDA_FDU_D6_3,DDA_FDU_F7_3,DDA_FDU_M8_3,DIA_FDU_D5_1,DIA_FDU_D6_1,DIA_FDU_F7_1,DIA_FDU_M8_1,DIA_FDU_D5_2,DIA_FDU_D6_2,DIA_FDU_F7_2,DIA_FDU_M8_2,DIA_FDU_D5_3,DIA_FDU_D6_3,DIA_FDU_F7_3,DIA_FDU_M8_3,DDA_NVG_D5_1,DDA_NVG_D6_1,DDA_NVG_F7_1,DDA_NVG_M8_1,DDA_NVG_D5_2,DDA_NVG_D6_2,DDA_NVG_F7_2,DDA_NVG_M8_2,DDA_NVG_D5_3,DDA_NVG_D6_3,DDA_NVG_F7_3,DDA_NVG_M8_3,DIA_BGI_D5_1,DIA_BGI_D6_1,DIA_BGI_F7_1,DIA_BGI_M8_1,DIA_BGI_D5_2,DIA_BGI_D6_2,DIA_BGI_F7_2,DIA_BGI_M8_2,DIA_BGI_D5_3,DIA_BGI_D6_3,DIA_BGI_F7_3,DIA_BGI_M8_3
rowname,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Asparagine synthetase [glutamine-hydrolyzing],19.860845,19.991346,20.145889,19.806990,19.967013,19.970993,20.093606,19.614526,19.995608,20.100427,20.086355,19.869796,19.775381,19.839469,20.209254,19.577877,19.803350,20.077635,20.342796,19.759229,19.909174,20.171098,20.099340,19.938791,19.891372,20.064317,20.181326,19.720069,19.803541,20.014973,20.275512,19.807146,19.976571,20.035710,20.141858,19.590997,19.684810,20.054783,20.251451,19.744597,19.909789,20.068129,20.310009,19.665728,19.856194,20.014483,20.235010,19.708410,19.821773,19.921400,20.153889,19.807108,19.896960,19.645544,20.329848,19.809738,20.174417,20.004120,20.221917,19.716678,19.913287,20.129740,20.225547,19.694733,19.920845,19.894252,20.270235,19.624782,19.903246,20.056401,20.243377,19.626948
7-dehydrocholesterol reductase,18.558493,18.917571,18.991289,18.684918,18.662426,19.032271,18.860557,18.776803,18.786864,18.800642,18.801431,19.100517,18.839435,18.531113,19.091305,18.845724,18.814530,18.784240,19.031933,18.789758,18.753161,18.570840,19.038279,18.883462,18.719959,18.756354,18.988160,18.724251,18.780447,18.896590,18.842797,18.833307,18.756866,18.751393,18.967301,18.956356,18.675268,18.808181,18.967849,18.892759,18.830904,18.606145,18.903201,18.899269,18.775478,18.866127,18.898324,18.850276,18.683514,18.480757,18.859581,18.580172,18.529426,19.271028,19.027993,18.768975,18.807566,19.001903,18.993718,18.969148,18.594127,18.767989,18.906309,18.685014,18.699358,18.930630,19.169211,18.738473,18.543816,18.986024,18.912861,19.039970
Glycolipid transfer protein,17.983987,18.928274,18.020263,19.044653,17.852047,18.577714,17.479855,18.915431,17.890074,18.969857,18.940817,18.783962,17.492014,18.751758,18.330559,18.941449,18.162516,18.723220,18.382085,18.758840,18.584169,18.153339,18.701750,18.405237,18.398640,18.588739,18.270514,18.592130,18.193263,18.321516,18.477697,18.834651,18.359653,18.292609,18.513505,18.544018,18.271716,18.361208,18.414510,18.827920,18.396416,18.503304,18.539536,18.829533,18.214655,18.360058,18.139170,18.528908,18.333994,18.297549,17.734340,19.239760,NaN,18.196135,18.497873,18.457580,18.243017,19.292838,18.182250,18.727757,18.002655,18.621614,18.356020,18.532285,18.375956,18.465961,18.227445,18.867166,18.370531,18.224416,18.542546,18.800341
"39S ribosomal protein L44, mitochondrial",19.098223,18.658105,18.812193,18.660543,18.778211,18.610128,18.676340,18.583907,19.338983,18.658374,18.883156,18.526606,19.179880,18.942425,18.468377,18.488505,19.441536,18.823241,18.230636,18.761014,18.920199,18.750227,18.749834,18.528896,18.800213,18.675607,18.640373,18.963697,18.776979,18.722274,18.608510,18.699248,19.253732,18.985702,18.503170,18.655266,18.701337,19.181015,18.649929,18.671866,19.089580,18.739683,18.771583,18.447250,19.105507,18.721854,18.793255,18.411912,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.850272,18.797546,18.703006,18.687197,19.057396,18.734226,18.796260,18.571871,19.017570,18.786674,18.623568,18.659185
Protein deglycase DJ-1,21.501443,

## Verify outputs

In [94]:
for fname in ['FedApp_corrected_data.tsv', 'FedApp_corrected_data_smpc.tsv']:
    fpath = os.path.join(after_dir, fname)
    if os.path.exists(fpath):
        df = pd.read_csv(fpath, sep='\t', index_col=0)
        print(f'{fname}: {df.shape[0]} features x {df.shape[1]} samples')
    else:
        print(f'{fname}: NOT FOUND')

# Check individual results
ind_dir = os.path.join(after_dir, 'individual_results')
if os.path.exists(ind_dir):
    for center in centers:
        for fname in ['full_corrected_data.csv', 'only_batch_corrected_data.csv', 'report.txt']:
            fpath = os.path.join(ind_dir, center, fname)
            status = 'OK' if os.path.exists(fpath) else 'MISSING'
            print(f'  {center}/{fname}: {status}')

FedApp_corrected_data.tsv: 3407 features x 72 samples
FedApp_corrected_data_smpc.tsv: 3407 features x 72 samples
  APT/full_corrected_data.csv: OK
  APT/only_batch_corrected_data.csv: OK
  APT/report.txt: OK
  FDU/full_corrected_data.csv: OK
  FDU/only_batch_corrected_data.csv: OK
  FDU/report.txt: OK
  NVG/full_corrected_data.csv: OK
  NVG/only_batch_corrected_data.csv: OK
  NVG/report.txt: OK
  BGI/full_corrected_data.csv: OK
  BGI/only_batch_corrected_data.csv: OK
  BGI/report.txt: OK


## Diagnostic: Compare central vs FedRBE betas

Verify that FedRBE and central R produce identical corrections.

**Known issue (fixed in PR):** The coordinator in `states.py` used a Python `set()` to
store the intersection of covariate hashes. Because each Docker container has a different
`PYTHONHASHSEED`, `set` iteration order differs between containers, causing each client
to build its design matrix with covariates in a different column order. When `XtX` matrices
are aggregated, the misaligned covariate columns produce wrong batch betas — but **only**
for features with NAs (where the balanced design is broken by missing samples).

The fix: use an order-preserving `list` instead of `set` for `global_variables_hashed`.

In [95]:
"""
Diagnostic: verify design matrix column alignment across clients,
simulate FedRBE locally, and compare with the actual Docker-based output
and with the central R correction.
"""
import sys, os
import pandas as pd, numpy as np
from numpy import linalg

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
from batchcorrection.classes.coordinator_utils import create_beta_mask

# --- 1. Check design matrix alignment from report files ---
print("=== Design matrix column alignment check ===")
ind_results = os.path.join(after_dir, 'individual_results')
col_orders = {}
for c in centers:
    report_path = os.path.join(ind_results, c, 'report.txt')
    if not os.path.exists(report_path):
        print(f"  {c}: report.txt not found, skipping alignment check")
        continue
    with open(report_path) as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        if 'corresponding design matrix was' in line.lower():
            col_orders[c] = lines[i+1].strip()
            break

if col_orders:
    # Check if all clients have the same column header
    unique_headers = set(col_orders.values())
    if len(unique_headers) == 1:
        print("  OK: All clients have identical design matrix columns")
    else:
        print("  WARNING: Design matrix columns DIFFER across clients!")
        for c, header in col_orders.items():
            print(f"    {c}: {header}")
        print("  This is the root cause: PYTHONHASHSEED differs across Docker")
        print("  containers, so set() iteration gives different covariate order.")

# --- 2. Load data ---
central_corr = pd.read_csv(os.path.join(after_dir, 'intensities_log_Rcorrected_UNION.tsv'), sep='\t', index_col=0)
fed_corr = pd.read_csv(os.path.join(after_dir, 'FedApp_corrected_data.tsv'), sep='\t', index_col=0)

per_center_data = {}; per_center_design = {}
for c in centers:
    per_center_data[c] = pd.read_csv(os.path.join(before_dir, c, 'intensities_log_UNION.tsv'), sep='\t', index_col=0)
    per_center_design[c] = pd.read_csv(os.path.join(before_dir, c, 'design.tsv'), sep='\t', index_col=0)

# --- 3. Build cohorts_order (must match FedRBE's coordinator) ---
cohorts_order = []
for c in centers:
    batches = sorted(per_center_design[c]['batch'].unique())
    cohorts_order.extend([f'{c}|{b}' for b in batches])
ref_batch = cohorts_order[-1]
design_cohorts = cohorts_order[:-1]
print(f'\nCohorts order: {cohorts_order}')
print(f'Reference: {ref_batch}')

# --- 4. Global feature set ---
all_features = set()
for c in centers:
    all_features.update(per_center_data[c].dropna(how='all').index)
global_features = sorted(all_features)
n_feat = len(global_features)
k = 1 + 3 + len(design_cohorts)
print(f'Global features: {n_feat}, Design columns: {k}')

# --- 5. Build feature_presence_matrix ---
feat2idx = {f: i for i, f in enumerate(global_features)}
fpm = np.zeros((n_feat, len(cohorts_order)), dtype=int)
for c in centers:
    d = per_center_data[c].dropna(how='all')
    des = per_center_design[c]
    for batch_name in sorted(des['batch'].unique()):
        batch_label = f'{c}|{batch_name}'
        co_idx = cohorts_order.index(batch_label)
        batch_samples = des[des['batch'] == batch_name].index
        for feat in d.index:
            if not d.loc[feat, batch_samples].isnull().all():
                fpm[feat2idx[feat], co_idx] = 1

# --- 6. Create beta mask ---
global_mask = create_beta_mask(fpm, n_feat, k)

# --- 7. Compute XtX/XtY per client, aggregate ---
XtX_g = np.zeros((n_feat, k, k))
XtY_g = np.zeros((n_feat, k))
for c in centers:
    d = per_center_data[c]
    des = per_center_design[c]
    is_ref = (c == centers[-1])
    n_s = len(des)
    X = np.zeros((n_s, k))
    X[:, 0] = 1
    X[:, 1] = des['D6'].values
    X[:, 2] = des['F7'].values
    X[:, 3] = des['M8'].values
    for di, dc in enumerate(design_cohorts):
        ci = 4 + di
        dc_client, dc_batch = dc.split('|')
        if is_ref:
            X[:, ci] = -1
        elif dc_client == c:
            X[:, ci] = (des['batch'] == dc_batch).astype(float)
    for fi, feat in enumerate(global_features):
        if feat not in d.index:
            continue
        y = d.loc[feat].values.astype(float)
        nn = np.isfinite(y)
        if nn.sum() > 0:
            Xn, yn = X[nn], y[nn]
            XtX_g[fi] += Xn.T @ Xn
            XtY_g[fi] += Xn.T @ yn

# --- 8. Solve betas ---
beta_sim = np.zeros((n_feat, k))
for i in range(n_feat):
    mask = global_mask[i]
    keep = ~mask
    sub = XtX_g[i][np.ix_(keep, keep)]
    d = linalg.det(sub)
    if abs(d) < 1e-10:
        beta_sim[i, keep] = linalg.pinv(sub) @ XtY_g[i, keep]
    else:
        beta_sim[i, keep] = linalg.solve(sub, XtY_g[i, keep])

# --- 9. Compute corrections from simulated betas ---
sim_corrected = {}
for c in centers:
    d = per_center_data[c]
    des = per_center_design[c]
    is_ref = (c == centers[-1])
    n_s = len(des)
    X_batch = np.zeros((n_s, len(design_cohorts)))
    for di, dc in enumerate(design_cohorts):
        dc_client, dc_batch = dc.split('|')
        if is_ref:
            X_batch[:, di] = -1
        elif dc_client == c:
            X_batch[:, di] = (des['batch'] == dc_batch).astype(float)
    for feat in d.index:
        if feat not in feat2idx:
            continue
        fi = feat2idx[feat]
        batch_betas = beta_sim[fi, 4:]
        be = batch_betas @ X_batch.T
        y = d.loc[feat].values.astype(float)
        corrected = np.where(np.isfinite(y), y - be, np.nan)
        for si, sample in enumerate(des.index):
            sim_corrected.setdefault(feat, {})[sample] = corrected[si]

# --- 10. Compare: central vs simulated vs actual FedRBE ---
common_features = central_corr.index.intersection(fed_corr.index)
common_samples = central_corr.columns.intersection(fed_corr.columns)

diffs_sim_vs_central = []
diffs_fed_vs_central = []
diffs_sim_vs_fed = []
for feat in common_features:
    for sample in common_samples:
        cv = central_corr.loc[feat, sample]
        fv = fed_corr.loc[feat, sample]
        sv = sim_corrected.get(feat, {}).get(sample, np.nan)
        if np.isfinite(cv) and np.isfinite(fv) and np.isfinite(sv):
            diffs_sim_vs_central.append(abs(cv - sv))
            diffs_fed_vs_central.append(abs(cv - fv))
            diffs_sim_vs_fed.append(abs(fv - sv))

dsc = np.array(diffs_sim_vs_central)
dfc = np.array(diffs_fed_vs_central)
dsf = np.array(diffs_sim_vs_fed)

print(f'\n=== Comparison summary (all features) ===')
print(f'Simulated vs Central: max={dsc.max():.2e}, mean={dsc.mean():.2e}')
print(f'FedRBE    vs Central: max={dfc.max():.2e}, mean={dfc.mean():.2e}')
print(f'Simulated vs FedRBE:  max={dsf.max():.2e}, mean={dsf.mean():.2e}')

if dsf.max() > 1e-6:
    print(f'\n*** Simulation matches Central but NOT actual FedRBE output! ***')
    print(f'Root cause: PYTHONHASHSEED differs across Docker containers,')
    print(f'causing set() iteration to give different covariate column orders.')
    print(f'Fix: use sorted list instead of set in states.py (see batchcorrection/states.py)')
elif dfc.max() > 1e-6:
    print(f'\n*** Simulation does NOT match Central — algorithm difference detected ***')
else:
    print(f'\n*** All three match — no discrepancy detected ***')

=== Design matrix column alignment check ===


  OK: All clients have identical design matrix columns

Cohorts order: ['APT|DDA_APT', 'APT|DIA_APT', 'FDU|DDA_FDU', 'FDU|DIA_FDU', 'NVG|DDA_NVG', 'BGI|DIA_BGI']
Reference: BGI|DIA_BGI
Global features: 3407, Design columns: 9

=== Comparison summary (all features) ===
Simulated vs Central: max=1.24e-13, mean=2.97e-14
FedRBE    vs Central: max=1.85e-13, mean=2.99e-14
Simulated vs FedRBE:  max=1.03e-13, mean=2.29e-15

*** All three match — no discrepancy detected ***


## Cleanup: remove tests and logs folders, stop controller, prune Docker volumes

In [97]:
# delete all report.txt
for c in centers:
    report_path = os.path.join(ind_results, c, 'report.txt')
    if os.path.exists(report_path):
        os.remove(report_path)

In [96]:
import shutil
import subprocess
from FeatureCloud.api.imp.controller import commands as fc_controller

# Stop the FeatureCloud controller first (so it releases file locks)
try:
    fc_controller.stop(name=fc_controller.DEFAULT_CONTROLLER_NAME)
    print('FeatureCloud controller stopped.')
except Exception as e:
    print(f'Controller stop: {e}')

# Remove tests and logs folders using Docker (files are owned by root from container runs)
tests_dir = os.path.join(multibatch_dir, 'tests')
logs_dir = os.path.join(multibatch_dir, 'logs')
workflows_dir = os.path.join(multibatch_dir, 'workflows')

for folder in [tests_dir, logs_dir, workflows_dir]:
    if os.path.exists(folder):
        # Use a Docker container to delete contents as root (rm the contents, not the mount point)
        subprocess.run(
            ['docker', 'run', '--rm', '-v', f'{folder}:/cleanup', 'alpine', 'sh', '-c', 'rm -rf /cleanup/*'],
            check=True,
        )
        # Now the folder is empty and owned by current user, safe to remove
        shutil.rmtree(folder, ignore_errors=True)
        print(f'Removed: {folder}')
    else:
        print(f'Not found (already clean): {folder}')

# Remove dangling Docker volumes created by tests
import docker
client = docker.from_env()
pruned = client.volumes.prune()
freed = pruned.get('SpaceReclaimed', 0)
print(f'Pruned Docker volumes. Space reclaimed: {freed / 1e9:.2f} GB')

# plus remove site_info.json if exists
site_info_path = os.path.join(multibatch_dir, 'site_info.json')
if os.path.exists(site_info_path):
    os.remove(site_info_path)
    print(f'Removed: {site_info_path}')
else:
    print(f'Not found (already clean): {site_info_path}') 

FeatureCloud controller stopped.
Removed: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/tests
Removed: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/logs
Removed: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/workflows
Pruned Docker volumes. Space reclaimed: 0.00 GB
Not found (already clean): /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/site_info.json
